# Deforestation from space: MODIS NDVI via GIBS

**Goal:** show *where* and *when* tropical forest was cleared between 2001 and 2022, and one consequence
(cleared land gets hotter). It uses the same GIBS helpers as `exploration.ipynb`.

**Why NDVI can show deforestation**
- NDVI measures how green vegetation is: near 0 for bare ground, above 0.7 for dense rainforest (Phase 1 check).
- Intact rainforest stays green through the dry season because deep roots reach groundwater.
- Cleared land (pasture, cropland, plantations) dries out in the dry season, so its NDVI drops.
- So **a lasting drop in dry-season NDVI** at a pixel is a strong hint that the forest there was cleared.
  Section 5 checks how big that drop is in the actual data before using it as a threshold.

**Figures in this notebook**

| § | Figure | Question it answers |
|---|---|---|
| 2 | Overview change map | Where in the Amazon did dry-season greenness fall? |
| 4 | Before / after / difference maps | What does clearing look like up close in each hotspot? |
| 5 | Histograms of NDVI change | How big a drop counts as "cleared"? |
| 6 | Year-of-clearing maps | When was each pixel cleared? |
| 7 | Cumulative cleared area | How fast did each region lose forest? |
| 8 | Terra vs. Aqua NDVI | Does orbit drift affect NDVI after 2022? |
| 9 | Year × month heatmap | How does the seasonal cycle change as forest disappears? |
| 10 | NDVI change vs. temperature change | Does clearing warm the land surface? |

**Runtime:** the first run makes roughly 800 GIBS requests (mostly small regional images), which can take
10–20 minutes. Everything is cached under `data/cache/`, so later runs take seconds.

## 0. Setup (copied from `exploration.ipynb`)

These cells are **copied unchanged** from Phase 1 so this notebook runs on its own.
Two demo calls at the ends of cells were removed (`find_layers("snow")` and the `load_colormap` example).
If you change a helper in `exploration.ipynb`, copy the change here as well.

In [ ]:
import io
import re
import time
import hashlib
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image

GIBS = "https://gibs.earthdata.nasa.gov"
WMS_URL = f"{GIBS}/wms/epsg4326/best/wms.cgi"
WMTS_CAPS_URL = f"{GIBS}/wmts/epsg4326/best/1.0.0/WMTSCapabilities.xml"

CACHE = Path("data/cache")
CACHE.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers["User-Agent"] = "dsc106-climate-exploration/0.1"

pd.set_option("display.max_colwidth", 80)

In [ ]:
NS = {
    "w": "http://www.opengis.net/wmts/1.0",
    "ows": "http://www.opengis.net/ows/1.1",
    "xlink": "http://www.w3.org/1999/xlink",
}
HREF = f"{{{NS['xlink']}}}href"


def cached_get(url, path, refresh=False, **params):
    """GET `url` once and keep the bytes at `path`."""
    path = Path(path)
    if path.exists() and not refresh:
        return path.read_bytes()
    r = session.get(url, params=params or None, timeout=120)
    r.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(r.content)
    return r.content


def load_catalog(refresh=False):
    xml = cached_get(WMTS_CAPS_URL, CACHE / "WMTSCapabilities.xml", refresh=refresh)
    root = ET.fromstring(xml)
    rows = []
    for layer in root.iter(f"{{{NS['w']}}}Layer"):
        hrefs = [m.get(HREF) or "" for m in layer.findall("ows:Metadata", NS)]
        rows.append({
            "layer": layer.find("ows:Identifier", NS).text,
            "title": layer.find("ows:Title", NS).text,
            "format": layer.find("w:Format", NS).text,
            "resolution": layer.find("w:TileMatrixSetLink/w:TileMatrixSet", NS).text,
            "time_ranges": [v.text for v in layer.findall("w:Dimension/w:Value", NS)],
            "colormap_url": next((h for h in hrefs if "/colormaps/v1.3/" in h), None),
            "legend_url": next((lg.get(HREF) for lg in layer.iter(f"{{{NS['w']}}}LegendURL")
                                if lg.get(HREF, "").endswith("_H.svg")), None),
        })
    return pd.DataFrame(rows).set_index("layer")


catalog = load_catalog()
modis = catalog[catalog.index.str.startswith("MODIS_")].copy()
print(f"{len(catalog)} GIBS layers, {len(modis)} MODIS layers")

In [ ]:
PERIOD_RE = re.compile(r"^P(\d+)([DMY])$")


def expand_range(spec):
    """'2000-03-01/2000-06-01/P1M' -> DatetimeIndex of every date in the range."""
    parts = spec.split("/")
    if len(parts) == 1:
        return pd.DatetimeIndex([parts[0][:10]])
    start, end, period = parts
    m = PERIOD_RE.match(period)
    if m is None:                       # sub-daily periods (PT10M etc.): just keep the endpoints
        return pd.DatetimeIndex([start[:10], end[:10]])
    n, unit = int(m.group(1)), m.group(2)
    freq = {"D": f"{n}D", "M": f"{n}MS", "Y": f"{n}YS"}[unit]
    return pd.date_range(start[:10], end[:10], freq=freq)


def layer_dates(layer):
    ranges = catalog.loc[layer, "time_ranges"]
    return pd.DatetimeIndex(sorted(set().union(*[expand_range(r) for r in ranges])))


def summarize(layer):
    d = layer_dates(layer)
    period = catalog.loc[layer, "time_ranges"][0].split("/")[-1]
    out = {"first": d.min().date(), "last": d.max().date(), "n_dates": len(d), "period": period}
    if period == "P1M":                 # count missing months inside the span
        full = pd.date_range(d.min(), d.max(), freq="MS")
        missing = full.difference(d)
        out["n_missing"] = len(missing)
        out["missing_example"] = ", ".join(m.strftime("%Y-%m") for m in missing[:4]) + (" …" if len(missing) > 4 else "")
    return out

In [ ]:
def find_layers(pattern):
    hits = modis[modis.index.str.contains(pattern, case=False) | modis["title"].str.contains(pattern, case=False)]
    return hits[["title", "format", "resolution"]].assign(
        first=[layer_dates(l).min().date() for l in hits.index],
        last=[layer_dates(l).max().date() for l in hits.index],
    )

In [ ]:
NUM = r"([+-]?(?:INF|\d*\.?\d+(?:[eE][+-]?\d+)?))"
RANGE_RE = re.compile(rf"^\s*[\[(]{NUM}\s*,\s*{NUM}\s*[\])]\s*$")
SINGLE_RE = re.compile(rf"^\s*\[?{NUM}\]?\s*$")


def _num(s):
    s = s.upper()
    return float("inf") if s in ("INF", "+INF") else float("-inf") if s == "-INF" else float(s)


def rgb_key(r, g, b):
    return (np.asarray(r, np.int64) << 16) | (np.asarray(g, np.int64) << 8) | np.asarray(b, np.int64)


@dataclass
class Colormap:
    title: str
    units: str | None
    keys: np.ndarray          # packed RGB, sorted
    values: np.ndarray        # decoded value per key (NaN = nodata / class)
    classes: dict = field(default_factory=dict)   # packed RGB -> label for classification colors
    n_numeric: int = 0

    def __repr__(self):
        lo, hi = np.nanmin(self.values), np.nanmax(self.values)
        return (f"Colormap({self.title!r}, units={self.units!r}, {self.n_numeric} numeric bins "
                f"[{lo:g} … {hi:g}], {len(self.classes)} classes)")


def parse_colormap(xml_bytes):
    root = ET.fromstring(xml_bytes)
    keys, values, classes = [], [], {}
    title = units = None
    numeric = []                                   # (index, lo, hi) to fix up edge bins afterwards
    for cm in root.iter("ColorMap"):
        tooltips = {le.get("id"): le.get("tooltip") for le in cm.iter("LegendEntry")}
        for e in cm.iter("ColorMapEntry"):
            key = int(rgb_key(*map(int, e.get("rgb").split(","))))
            v = e.get("value")
            if e.get("nodata") == "true" or v is None:
                keys.append(key); values.append(np.nan)
                classes[key] = e.get("label") or tooltips.get(e.get("ref")) or ("No Data" if e.get("nodata") == "true" else "class")
                continue
            if title is None:
                title, units = cm.get("title"), cm.get("units")
            if (m := RANGE_RE.match(v)):
                lo, hi = _num(m.group(1)), _num(m.group(2))
                numeric.append((len(values), lo, hi))
                val = lo if np.isinf(hi) else hi if np.isinf(lo) else (lo + hi) / 2
            elif (m := SINGLE_RE.match(v)):
                val = _num(m.group(1))
            else:
                raise ValueError(f"unrecognized colormap value {v!r}")
            keys.append(key); values.append(val)
    # first/last numeric range bins are catch-alls: use their inner edge
    if len(numeric) >= 2:
        numeric.sort(key=lambda t: t[1])
        (i0, _, hi0), (i1, lo1, _) = numeric[0], numeric[-1]
        values[i0], values[i1] = (hi0 if np.isfinite(hi0) else values[i0]), (lo1 if np.isfinite(lo1) else values[i1])
    keys, values = np.array(keys, np.int64), np.array(values, float)
    order = np.argsort(keys, kind="stable")
    keys, idx = np.unique(keys[order], return_index=True)   # keep first entry if a color repeats
    values = values[order][idx]
    return Colormap(title, units, keys, values, classes, int(np.isfinite(values).sum()))


def load_colormap(layer):
    url = catalog.loc[layer, "colormap_url"]
    if url is None:
        raise ValueError(f"{layer} has no color map (it may be imagery or vector data)")
    return parse_colormap(cached_get(url, CACHE / "colormaps" / Path(url).name))

In [ ]:
GLOBAL = (-180, -90, 180, 90)


def fetch_png(layer, when, bbox=GLOBAL, width=720, height=None, refresh=False):
    """Return the WMS image as an (H, W, 4) uint8 RGBA array."""
    west, south, east, north = bbox
    if height is None:                               # keep pixels square in degrees
        height = max(1, round(width * (north - south) / (east - west)))
    when = pd.Timestamp(when).strftime("%Y-%m-%d")
    tag = hashlib.md5(f"{bbox}".encode()).hexdigest()[:8]
    path = CACHE / "wms" / layer / f"{when}_{tag}_{width}x{height}.png"
    if not path.exists() or refresh:
        params = dict(SERVICE="WMS", REQUEST="GetMap", VERSION="1.3.0", LAYERS=layer,
                      CRS="EPSG:4326", BBOX=f"{south},{west},{north},{east}",
                      WIDTH=width, HEIGHT=height, FORMAT="image/png", TIME=when)
        for attempt in range(3):
            r = session.get(WMS_URL, params=params, timeout=120)
            if r.ok and r.headers.get("content-type", "").startswith("image/"):
                break
            time.sleep(2 ** attempt)
        else:
            raise RuntimeError(f"GIBS error for {layer} @ {when}: {r.status_code} {r.text[:300]}")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    return np.array(Image.open(path).convert("RGBA"))


def decode(rgba, cmap):
    """Map RGBA pixels to values. Transparent / nodata / unmatched -> NaN."""
    key = rgb_key(rgba[..., 0], rgba[..., 1], rgba[..., 2])
    pos = np.clip(np.searchsorted(cmap.keys, key), 0, len(cmap.keys) - 1)
    matched = cmap.keys[pos] == key
    opaque = rgba[..., 3] > 0
    values = np.where(matched & opaque, cmap.values[pos], np.nan)
    unmatched = (opaque & ~matched).sum()
    return values, unmatched


@dataclass
class Grid:
    layer: str
    date: pd.Timestamp
    values: np.ndarray
    lats: np.ndarray       # pixel-center latitudes, north -> south
    lons: np.ndarray       # pixel-center longitudes, west -> east
    units: str | None = None
    title: str | None = None

    @property
    def extent(self):
        dlat, dlon = abs(self.lats[1] - self.lats[0]), abs(self.lons[1] - self.lons[0])
        return (self.lons[0] - dlon / 2, self.lons[-1] + dlon / 2, self.lats[-1] - dlat / 2, self.lats[0] + dlat / 2)

    def sel(self, lat=None, lon=None):
        """Crop to lat=(south, north), lon=(west, east)."""
        r = np.ones(len(self.lats), bool) if lat is None else (self.lats >= lat[0]) & (self.lats <= lat[1])
        c = np.ones(len(self.lons), bool) if lon is None else (self.lons >= lon[0]) & (self.lons <= lon[1])
        return Grid(self.layer, self.date, self.values[np.ix_(r, c)], self.lats[r], self.lons[c], self.units, self.title)

    def mean(self):
        """Area-weighted mean (cos latitude) over pixels with data."""
        w = np.broadcast_to(np.cos(np.deg2rad(self.lats))[:, None], self.values.shape)
        ok = np.isfinite(self.values)
        return float((self.values[ok] * w[ok]).sum() / w[ok].sum()) if ok.any() else np.nan

    def coverage(self):
        return float(np.isfinite(self.values).mean())

    def plot(self, ax=None, **kw):
        ax = ax or plt.gca()
        im = ax.imshow(self.values, extent=self.extent, origin="upper", interpolation="nearest", **kw)
        ax.set_title(f"{self.title or self.layer}\n{self.date:%Y-%m-%d}", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.7, label=self.units or "")
        return im


_cmaps = {}

def get_grid(layer, when, bbox=GLOBAL, width=720, height=None, refresh=False):
    if layer not in _cmaps:
        _cmaps[layer] = load_colormap(layer)
    cmap = _cmaps[layer]
    rgba = fetch_png(layer, when, bbox, width, height, refresh)
    values, unmatched = decode(rgba, cmap)
    if unmatched:
        print(f"warning: {unmatched} opaque pixels in {layer} @ {when} did not match the color map")
    h, w = values.shape
    west, south, east, north = bbox
    lats = north - (np.arange(h) + 0.5) * (north - south) / h
    lons = west + (np.arange(w) + 0.5) * (east - west) / w
    return Grid(layer, pd.Timestamp(when), values, lats, lons, cmap.units, cmap.title)

## 1. Layers, regions and settings

### Layers

- **Terra NDVI (monthly)** is the main signal. It starts in March 2000.
- **Aqua NDVI** is used only in §8, to check orbit drift. It starts in July 2002.
- **Terra daytime land surface temperature (LST)** is used in §10 to test whether cleared land warmed up.

The table below confirms each layer's date range and native resolution. **Check the `resolution` column:**
the settings cell assumes NDVI at about 1 km and LST at about 5 km (0.05°).

In [ ]:
import warnings
from matplotlib.colors import BoundaryNorm, ListedColormap, to_rgba
from matplotlib.patches import Rectangle
import matplotlib.patheffects as pe

NDVI = "MODIS_Terra_L3_NDVI_Monthly"
NDVI_AQUA = "MODIS_Aqua_L3_NDVI_Monthly"
LST = "MODIS_Terra_L3_Land_Surface_Temp_Monthly_Day"
VEG = NDVI          # the vegetation layer used everywhere below

layers = [NDVI, NDVI_AQUA, LST]
pd.DataFrame({l: summarize(l) for l in layers}).T.assign(resolution=catalog.loc[layers, "resolution"])

**Optional, EVI instead of NDVI.** NDVI saturates over very dense forest: above about 0.8, more leaves barely
raise it. EVI does not saturate as much. To try it, run `find_layers("EVI")`, then set `VEG` to the Terra monthly
EVI layer and re-run from here. Cleared land is far below the saturation level, so NDVI already works for
detecting clearing. EVI would mainly sharpen differences *inside* intact forest.

### Study regions

Each region is a box of about 4° × 3–4° (roughly 400 × 400 km), requested at full resolution.

| Region | Why it's here | Dry-season months used |
|---|---|---|
| **Rondônia, Brazil** | Classic "fishbone" clearing along the BR-364 highway and its side roads | Jul–Sep |
| **Pará (BR-163), Brazil** | Clearing frontier along the Cuiabá–Santarém highway | Jul–Sep |
| **Riau, Sumatra** | Rainforest and peat swamp converted to oil palm and pulpwood plantations | Jun–Aug |
| **Paraguayan Chaco** | Dry forest cleared for cattle ranching | Jul–Sep |
| **Intact Amazon (control)** | Remote interior forest. It should show **no** trend, so any trend here is instrument noise or drift | Jul–Sep |

**Why only dry-season months**
- **Contrast:** forest stays green while pasture dries out, so the difference between them is largest then.
- **Clouds:** the wet season is cloudy, and cloudy pixels come back as NaN.

**Why the median of three months**
- One month can be spoiled by cloud or fire smoke.
- The median of three months ignores one bad month. With NaNs, it uses the months that remain.

In [ ]:
# bbox = (west, south, east, north); integer degrees so the pixel grids line up exactly
REGIONS = {
    "Rondônia":                dict(bbox=(-64, -12, -60, -8), months=(7, 8, 9)),
    "Pará (BR-163)":           dict(bbox=(-57, -9, -53, -5),  months=(7, 8, 9)),
    "Riau, Sumatra":           dict(bbox=(100, -1, 104, 2),   months=(6, 7, 8)),
    "Paraguayan Chaco":        dict(bbox=(-62, -23, -58, -20), months=(7, 8, 9)),
    "Intact Amazon (control)": dict(bbox=(-68, -6, -64, -2),  months=(7, 8, 9)),
}
CONTROL = "Intact Amazon (control)"
HOTSPOTS = [r for r in REGIONS if r != CONTROL]

DEG = 0.01                       # degrees per pixel for regional requests (~1.1 km, about NDVI's native size)
YEARS = range(2001, 2023)        # 2001–2022: the reliable pre-orbit-drift period (§8 checks NDVI specifically)
BASE_YEARS = range(2001, 2004)   # "before" = mean of 2001–2003
RECENT_YEARS = range(2020, 2023) # "after"  = mean of 2020–2022
DROP = 0.20                      # NDVI drop below baseline that counts as cleared (justified in §5)
PERSIST = 3                      # the drop must last this many years (fewer at the end of the record)
SEARCH_FROM = max(BASE_YEARS) + 1

yrs = np.array(list(YEARS))

### Plot style

**Why each color choice**
- **Regions keep one fixed color in every chart** (color follows the entity), so a reader learns it once.
  The four hotspot colors were checked to stay distinguishable for color-blind readers.
- **The control is gray**, because it is a reference line, not a fifth story.
- **NDVI maps use one hue (light → dark green)**, because NDVI is a magnitude.
- **Change maps use brown ↔ teal with a near-white middle**, because change has a direction:
  brown = less green, teal = greener, and zero sits at the neutral center.
- **No data (water, cloud) is mid-gray**, so it can't be mistaken for "no change".
- **Two lighter region colors (aqua and yellow) have weak contrast** against the background.
  That is why line charts always carry direct text labels as well as a legend.

In [ ]:
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
SURFACE, GRIDC, AXIS = "#fcfcfb", "#e1e0d9", "#c3c2b7"
NODATA, STABLE = "#c3c2b7", "#f0efec"

REGION_COLORS = {
    "Rondônia": "#2a78d6",          # blue
    "Pará (BR-163)": "#eb6834",     # orange
    "Riau, Sumatra": "#1baf7a",     # aqua
    "Paraguayan Chaco": "#eda100",  # yellow
    CONTROL: MUTED,                 # gray reference
}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10, "axes.titlelocation": "left", "font.size": 9,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRIDC, "grid.linewidth": 0.8, "lines.linewidth": 2,
    "legend.frameon": False, "figure.dpi": 110,
})


def with_nodata(name):
    cmap = plt.get_cmap(name).copy()
    cmap.set_bad(NODATA)
    return cmap

NDVI_CMAP = with_nodata("Greens")
DIFF_CMAP = with_nodata("BrBG")
TEMP_CMAP = with_nodata("RdBu_r")

### Helpers

| Helper | What it does |
|---|---|
| `dry_season(layer, year, months, bbox, deg)` | Median of that year's dry-season monthly grids, as a `Grid` |
| `window_mean(stack, years)` | Mean over a block of years in a (year, lat, lon) stack |
| `clearing_year(stack, base)` | First year each pixel drops `DROP` below baseline **and stays down** |
| `pixel_area_km2(grid)` | Area of each pixel (pixels shrink with latitude: about 111 km per degree × cos(latitude)) |
| `show_map(ax, values, extent, ...)` | `imshow` with the map styling used throughout |

`nanmean` and `nanmedian` are NumPy's versions with the "mean of empty slice" warning silenced.
That warning fires for every water pixel and would hide real messages.

In [ ]:
def nanmean(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a, axis=axis)


def nanmedian(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(a, axis=axis)


def width_for(bbox, deg=DEG):
    return round((bbox[2] - bbox[0]) / deg)


_dates = {}

def dry_season(layer, year, months, bbox, deg=DEG):
    # Median of the available dry-season months for one year -> Grid
    if layer not in _dates:
        _dates[layer] = set(layer_dates(layer))
    wanted = [pd.Timestamp(year, m, 1) for m in months]
    grids = [get_grid(layer, d, bbox, width_for(bbox, deg)) for d in wanted if d in _dates[layer]]
    if not grids:
        raise ValueError(f"{layer}: none of {[d.strftime('%Y-%m') for d in wanted]} available")
    values = nanmedian(np.stack([g.values for g in grids]), axis=0).astype(np.float32)
    g = grids[0]
    return Grid(layer, pd.Timestamp(year, 1, 1), values, g.lats, g.lons, g.units, g.title)


def window_mean(stack, years):
    return nanmean(stack[np.isin(yrs, list(years))], axis=0)


def clearing_year(stack, base, drop=DROP, persist=PERSIST, start=SEARCH_FROM):
    # First year where NDVI is `drop` below baseline and stays below for `persist` years.
    # Cloudy years (NaN) neither break nor count toward the run. Near the end of the record
    # fewer than `persist` years remain, so the last detections are less certain.
    below = stack < (base - drop)          # NaN compares as False
    unknown = np.isnan(stack)
    out = np.full(base.shape, np.nan, np.float32)
    for i in np.flatnonzero(yrs >= start):
        w = slice(i, min(i + persist, len(stack)))
        hit = below[i] & (below[w] | unknown[w]).all(axis=0) & np.isnan(out)
        out[hit] = yrs[i]
    out[~np.isfinite(base)] = np.nan
    return out


def pixel_area_km2(grid, deg=DEG):
    km = deg * 111.32
    return np.broadcast_to((km * km * np.cos(np.deg2rad(grid.lats)))[:, None], grid.values.shape)


def lon_fmt(v, _):
    return f"{abs(v):g}°{'W' if v < 0 else 'E' if v > 0 else ''}"


def lat_fmt(v, _):
    return f"{abs(v):g}°{'S' if v < 0 else 'N' if v > 0 else ''}"


def show_map(ax, values, extent, title=None, **kw):
    im = ax.imshow(values, extent=extent, origin="upper", interpolation="nearest", **kw)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(AXIS); s.set_linewidth(0.6)
    ax.xaxis.set_major_formatter(lon_fmt)
    ax.yaxis.set_major_formatter(lat_fmt)
    ax.tick_params(labelsize=7, length=2)
    if title:
        ax.set_title(title, fontsize=9)
    return im


def spread(values, gap):
    # Nudge label y-positions apart so end-of-line labels don't overlap (order preserved)
    order = np.argsort(values)
    out = np.array(values, float)
    for a, b in zip(order[:-1], order[1:]):
        if out[b] - out[a] < gap:
            out[b] = out[a] + gap
    return out

## 2. Overview: where did dry-season greenness fall?

**What it shows:** the change in dry-season NDVI across the southern Amazon, from 2001–03 to 2020–22.

**How it's made**
- Request the whole area at 0.05° (about 5.5 km). That is coarse enough to cover a large area in few requests.
- For each period, take the median of each year's Jul–Sep grids, then average the three years.
- Subtract "before" from "after".
- Requests: 2 periods × 3 years × 3 months = **18**.

**What to look for**
- Brown should concentrate along the southern and eastern edge of the forest.
  This is the "arc of deforestation" running through Rondônia, Mato Grosso and Pará.
- The interior should stay near white (no change).
- The boxes mark the regions studied in close-up below.

In [ ]:
OVERVIEW = (-75, -18, -45, 2)
OV_DEG = 0.05

def window_grid(layer, years, months, bbox, deg):
    gs = [dry_season(layer, y, months, bbox, deg) for y in years]
    return Grid(layer, gs[0].date, nanmean(np.stack([g.values for g in gs]), axis=0), gs[0].lats, gs[0].lons,
                gs[0].units, gs[0].title)

ov_before = window_grid(VEG, BASE_YEARS, (7, 8, 9), OVERVIEW, OV_DEG)
ov_after = window_grid(VEG, RECENT_YEARS, (7, 8, 9), OVERVIEW, OV_DEG)
ov_delta = ov_after.values - ov_before.values

fig, ax = plt.subplots(figsize=(9, 6.6), layout="constrained")
im = show_map(ax, ov_delta, ov_before.extent, cmap=DIFF_CMAP, vmin=-0.4, vmax=0.4)
ax.set_title(f"Change in dry-season NDVI, {BASE_YEARS[0]}–{str(BASE_YEARS[-1])[2:]} → "
             f"{RECENT_YEARS[0]}–{str(RECENT_YEARS[-1])[2:]}  (Jul–Sep, MODIS Terra)", fontsize=11)

w0, s0, e0, n0 = OVERVIEW
for name, r in REGIONS.items():
    w, s, e, n = r["bbox"]
    if w >= w0 and e <= e0 and s >= s0 and n <= n0:
        ax.add_patch(Rectangle((w, s), e - w, n - s, fill=False, ec=INK, lw=1.2))
        ax.text(w, n + 0.3, name, fontsize=8, color=INK, va="bottom",
                path_effects=[pe.withStroke(linewidth=3, foreground=SURFACE)])

cb = fig.colorbar(im, ax=ax, shrink=0.7, extend="both")
cb.set_label("NDVI change   (brown = less green · teal = greener · gray = water / no data)")
plt.show()

## 3. Build the regional dry-season stacks

**What this does:** builds one dry-season median NDVI grid per year for each region, at `DEG` = 0.01°
(about 1 km). The result is a `(year, lat, lon)` array per region.

**How much it downloads**
- 22 years × 3 months × 5 regions = **330 requests**.
- Each request is about 400 × 400 pixels.
- This is the slow cell on the first run.

**Why the coverage check**
- Coverage = the share of pixels with a value in each year's composite.
- Water is always missing, so compare years within a region, not across regions.
- A year far below its region's usual coverage was cloudy or smoky. Expect that in Riau in big fire years.
  Treat that year's numbers with care.

In [ ]:
stacks, templates = {}, {}
for name, r in REGIONS.items():
    t0 = time.time()
    grids = [dry_season(VEG, y, r["months"], r["bbox"]) for y in YEARS]
    stacks[name] = np.stack([g.values for g in grids])
    templates[name] = grids[0]
    print(f"{name:26s} {stacks[name].shape}   {time.time() - t0:6.1f}s")

coverage = pd.DataFrame({n: np.isfinite(s).mean(axis=(1, 2)) for n, s in stacks.items()}, index=yrs)
low = coverage.lt(coverage.median() - 0.10)          # 10+ points below that region's usual coverage
print("\nYears with unusually low coverage:")
for n in coverage:
    bad = coverage.index[low[n]].tolist()
    print(f"  {n:26s} {bad if bad else 'none'}")
coverage.T.round(2)

**Baseline, recent and change grids** (no downloads).
`base` is the 2001–03 mean, `recent` is the 2020–22 mean, and `delta = recent − base`.

In [ ]:
base = {n: window_mean(s, BASE_YEARS) for n, s in stacks.items()}
recent = {n: window_mean(s, RECENT_YEARS) for n, s in stacks.items()}
delta = {n: recent[n] - base[n] for n in stacks}

pd.DataFrame({
    "baseline NDVI": {n: nanmean(base[n]) for n in stacks},
    "recent NDVI": {n: nanmean(recent[n]) for n in stacks},
    "mean change": {n: nanmean(delta[n]) for n in stacks},
}).round(3)

## 4. Before, after, and the difference

**Layout:** one row per region.
- Columns 1–2: dry-season NDVI at the start and end of the period, on the same green scale.
- Column 3: the difference, on the same brown ↔ teal scale as §2.

Sharing scales within each column means the colors mean the same thing in every row, so rows can be compared.

**What to look for**
- **Rondônia:** the fishbone pattern. Main highways have side roads every few km, and clearing spreads outward from them.
- **Riau:** large rectangular blocks (industrial plantations).
- **Chaco:** a regular grid of paddocks separated by thin strips of leftover forest.
- **Control:** should look the same in both years, with a nearly blank difference panel.

In [ ]:
names = list(REGIONS)
fig, axes = plt.subplots(len(names), 3, figsize=(10.5, 3.3 * len(names)), layout="constrained")
for row, n in zip(axes, names):
    ext = templates[n].extent
    im_ndvi = show_map(row[0], base[n], ext, cmap=NDVI_CMAP, vmin=0.2, vmax=0.9)
    show_map(row[1], recent[n], ext, cmap=NDVI_CMAP, vmin=0.2, vmax=0.9)
    im_diff = show_map(row[2], delta[n], ext, cmap=DIFF_CMAP, vmin=-0.4, vmax=0.4)
    row[0].set_ylabel(n, fontsize=10, color=INK, labelpad=8)

for ax, t in zip(axes[0], [f"Before ({BASE_YEARS[0]}–{BASE_YEARS[-1]})",
                           f"After ({RECENT_YEARS[0]}–{RECENT_YEARS[-1]})", "Change (after − before)"]):
    ax.set_title(t, fontsize=11)

fig.colorbar(im_ndvi, ax=axes[:, :2], location="bottom", shrink=0.5, pad=0.01,
             label="Dry-season NDVI (gray = water / no data)")
fig.colorbar(im_diff, ax=axes[:, 2], location="bottom", shrink=0.9, pad=0.01, extend="both",
             label="NDVI change")
fig.suptitle("Dry-season greenness before and after two decades of land-use change", fontsize=13,
             x=0.02, ha="left", color=INK)
plt.show()

## 5. How big a drop counts as "cleared"?

**Why this step exists:** §6 needs a threshold, and picking one without looking at the data would be guesswork.

**What it shows:** the distribution of pixel NDVI change, one panel per region.
- **Intact forest** gives a narrow peak near 0. In the control region that peak is the noise level of the method.
- **Clearing** gives a long tail, or a second bump, to the left.

**How to choose the threshold**
- Put `DROP` past the left edge of the control's peak, so almost no control pixels are flagged.
- It should still sit inside the hotspots' tail.
- The text in each panel gives the share of pixels below the current threshold.
- If the control share isn't close to 0%, increase `DROP` in the settings cell and re-run from §5.

In [ ]:
bins = np.linspace(-0.6, 0.3, 91)
fig, axes = plt.subplots(len(names), 1, figsize=(8, 1.6 * len(names)), sharex=True, layout="constrained")
for ax, n in zip(axes, names):
    d = delta[n][np.isfinite(delta[n])]
    ax.hist(d, bins=bins, color=REGION_COLORS[n], ec=SURFACE, lw=0.3)
    ax.axvline(-DROP, color=INK, lw=1, ls="--")
    share = (d < -DROP).mean()
    ax.text(0.01, 0.85, n, transform=ax.transAxes, fontsize=9, color=INK, va="top")
    ax.text(0.01, 0.55, f"{share:.1%} of pixels below −{DROP}", transform=ax.transAxes, fontsize=8,
            color=INK2, va="top")
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
axes[0].set_title(f"Distribution of dry-season NDVI change per pixel (dashed line: threshold −{DROP})")
axes[-1].set_xlabel("NDVI change, after − before")
plt.show()

## 6. When was each pixel cleared?

**Rule:** a pixel is "cleared in year *Y*" if, in year *Y*:
1. its dry-season NDVI is at least `DROP` below its own 2001–03 baseline, **and**
2. it stays below that level for the next `PERSIST` − 1 years. Cloudy years count as unknown and are skipped.

**Why require the drop to persist**
- One dry year, a fire, or one bad composite can cause a temporary dip.
- A real clearing stays cleared. Pasture doesn't grow back into rainforest within a few years.

**Limits**
- Clearing **before 2004** falls inside the baseline and cannot be detected.
- **2021–22** have fewer following years to confirm against, so those detections are less certain.

**Color:** one hue, light (early) → dark (recent), because year is an ordered quantity.
Off-white = forest that never crossed the threshold. Gray = water / no data.

In [ ]:
cleared = {n: clearing_year(stacks[n], base[n]) for n in names}

first, last = SEARCH_FROM, yrs.max()
loss_years = np.arange(first, last + 1)
LOSS_CMAP = ListedColormap(plt.get_cmap("Reds")(np.linspace(0.3, 1.0, len(loss_years))))
LOSS_CMAP.set_bad((0, 0, 0, 0))
LOSS_NORM = BoundaryNorm(np.append(loss_years, last + 1) - 0.5, LOSS_CMAP.N)

def background(b):
    rgba = np.empty(b.shape + (4,))
    rgba[:] = to_rgba(NODATA)
    rgba[np.isfinite(b)] = to_rgba(STABLE)
    return rgba

fig, axes = plt.subplots(2, 3, figsize=(12, 8.2), layout="constrained")
for ax, n in zip(axes.flat, names):
    ext = templates[n].extent
    show_map(ax, background(base[n]), ext)
    im = show_map(ax, np.ma.masked_invalid(cleared[n]), ext, title=n, cmap=LOSS_CMAP, norm=LOSS_NORM)
key = axes.flat[-1]                       # spare panel holds the key
key.axis("off")
cax = key.inset_axes([0.08, 0.62, 0.84, 0.07])
cb = fig.colorbar(im, cax=cax, orientation="horizontal", ticks=loss_years[::3])
cb.set_label("Year cleared", color=INK2)
cb.ax.tick_params(labelsize=8)
key.legend(handles=[Rectangle((0, 0), 1, 1, fc=STABLE, ec=AXIS), Rectangle((0, 0), 1, 1, fc=NODATA)],
           labels=["Not flagged (forest or already open in 2001–03)", "Water / no data"],
           loc="upper left", bbox_to_anchor=(0.04, 0.45), fontsize=8)
fig.suptitle(f"Year each pixel was first cleared (dry-season NDVI ≥ {DROP} below 2001–03 and staying down)",
             fontsize=12, x=0.02, ha="left", color=INK)
plt.show()

## 7. How fast did each region lose forest?

**What it shows:** the share of each region's land area flagged as cleared by each year.
It is the cumulative version of the §6 maps.

**How it's computed**
- Each pixel is weighted by its true area (km² shrink with cos(latitude)).
- Only pixels with a valid baseline count, so water is excluded from the total.
- The control line shows the method's **false-positive rate**. It should stay near 0%.

**How to check it**
- The table gives cleared area in km².
- Compare the order of magnitude with published estimates, e.g. Global Forest Watch tree-cover-loss data for the same boxes.
- Expect differences: GFW uses 30 m Landsat, and this uses about 1 km, which misses small clearings.

In [ ]:
rows = {}
for n in names:
    area = pixel_area_km2(templates[n])
    valid = np.isfinite(base[n])
    cy = cleared[n]
    rows[n] = {y: area[cy <= y].sum() / area[valid].sum() * 100 for y in range(first - 1, last + 1)}
share = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
ax.axvspan(last - 1.5, last + 0.5, color=GRIDC, alpha=0.6, lw=0)
ax.text(last - 0.5, 0.02, "fewer years\nto confirm", transform=ax.get_xaxis_transform(), fontsize=7,
        color=MUTED, va="bottom", ha="center")
for n in names:
    ls = "--" if n == CONTROL else "-"
    ax.plot(share.index, share[n], color=REGION_COLORS[n], ls=ls, label=n)
ends = share.iloc[-1]
ys = spread(ends.values, gap=share.values.max() * 0.045)
for n, y in zip(ends.index, ys):
    ax.annotate(f"{n}  {ends[n]:.1f}%", (last, ends[n]), xytext=(last + 0.7, y), textcoords="data",
                fontsize=8, color=INK2, va="center",
                arrowprops=dict(arrowstyle="-", color=REGION_COLORS[n], lw=1))
ax.set_xlim(first - 1, last + 6)
ax.set_xticks(range(first - 1, last + 1, 3))
ax.set_ylim(0, None)
ax.set_ylabel("% of land area cleared since 2001–03")
ax.grid(axis="y")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Cumulative area flagged as cleared, by region", fontsize=11)

plt.show()

summary = pd.DataFrame({
    "land area (km²)": {n: pixel_area_km2(templates[n])[np.isfinite(base[n])].sum() for n in names},
    f"cleared by {last} (km²)": {n: pixel_area_km2(templates[n])[cleared[n] <= last].sum() for n in names},
    f"% cleared by {last}": share.iloc[-1],
}).round(1)
summary

## 8. Orbit-drift check for NDVI

**Why:** Phase 1 found that Terra and Aqua drifted after 2022 and that land surface temperature changes because of it.
It also flagged that NDVI should be checked the same way.

**How**
- Use the **intact control region**, where nothing should change.
- Compare Terra and Aqua dry-season NDVI from 2003 to the latest complete dry season.
- 0.05° is enough for a regional mean, so requests are small (about 140).

**How to read it**
- **Both lines flat and close together after 2022:** NDVI is safe to extend past 2022.
  Change `YEARS` and `RECENT_YEARS` and re-run from §3.
- **The lines split or shift after 2022:** keep the analysis at 2022, as it is now.

In [ ]:
ctrl = REGIONS[CONTROL]
end = min(layer_dates(NDVI).max(), layer_dates(NDVI_AQUA).max())
last_full = end.year if end >= pd.Timestamp(end.year, max(ctrl["months"]), 1) else end.year - 1
drift_years = range(2003, last_full + 1)

drift = pd.DataFrame({
    sat: {y: dry_season(L, y, ctrl["months"], ctrl["bbox"], deg=0.05).mean() for y in drift_years}
    for sat, L in {"Terra": NDVI, "Aqua": NDVI_AQUA}.items()
})

fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
ax.axvspan(2022.5, last_full + 0.5, color=GRIDC, alpha=0.6, lw=0)
ax.text(2022.7, 0.97, "orbit drift", transform=ax.get_xaxis_transform(), fontsize=8, color=MUTED, va="top")
for sat, c in [("Terra", "#2a78d6"), ("Aqua", "#eb6834")]:
    ax.plot(drift.index, drift[sat], color=c, marker="o", ms=4, label=sat)
    ax.annotate(sat, (drift.index[-1], drift[sat].iloc[-1]), xytext=(6, 0), textcoords="offset points",
                fontsize=8, color=INK2, va="center")
ax.set_ylabel("Mean dry-season NDVI")
ax.grid(axis="y")
ax.legend(loc="lower left", fontsize=8)
mon = lambda m: pd.Timestamp(2000, m, 1).strftime("%b")
ax.set_title(f"Intact-forest control: Terra vs. Aqua dry-season NDVI ({mon(ctrl['months'][0])}–{mon(ctrl['months'][-1])})",
             fontsize=11)

plt.show()

drift.assign(diff=drift["Terra"] - drift["Aqua"]).loc[2018:].round(3)

## 9. How the seasonal cycle changes: Rondônia, year × month

**What it shows**
- Each cell is Rondônia's mean NDVI for one month, minus that calendar month's 2001–03 average.
- Rows are years and columns are months.
- Comparing each month with its own baseline removes the normal seasonal cycle, so only the change is left.

**What to look for**
- If clearing is the main signal, the **dry-season columns (Jul–Sep) should turn brown over time**, more than the wet-season columns.
  Pasture loses most of its greenness in the dry season.
- Wet-season months (Dec–Apr) are noisier because of clouds. This is why §3–7 use the dry season only.
- Unusually brown single years can point to droughts or big fire seasons. Check them against news or fire data before calling them that.

**Requests:** 22 years × 12 months at 0.05° = **264** small images.

In [ ]:
R = REGIONS["Rondônia"]
months_all = [d for d in layer_dates(VEG) if d.year in YEARS]
ron = pd.Series({d: get_grid(VEG, d, R["bbox"], width_for(R["bbox"], 0.05)).mean() for d in months_all})
tbl = ron.groupby([ron.index.year, ron.index.month]).first().unstack().reindex(columns=range(1, 13))
anom = tbl - tbl.loc[list(BASE_YEARS)].mean()

v = np.nanmax(np.abs(anom.values))
fig, ax = plt.subplots(figsize=(7, 7.5), layout="constrained")
im = ax.imshow(anom.values, cmap=DIFF_CMAP, vmin=-v, vmax=v, aspect="auto")
ax.set_xticks(range(12), list("JFMAMJJASOND"))
ax.set_yticks(range(len(anom)), anom.index)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.axvspan(5.5, 8.5, fill=False, ec=INK, lw=1)            # Jul–Sep outline
ax.text(7, -0.8, "dry season", ha="center", fontsize=8, color=INK2)
fig.colorbar(im, ax=ax, shrink=0.6, label="NDVI vs. 2001–03 same-month mean (gray = no data)")
ax.set_title("Rondônia: monthly NDVI anomaly", fontsize=11, pad=16)
plt.show()

## 10. Does clearing warm the land surface?

**Idea:** forests cool themselves by moving water from the soil into the air (evapotranspiration).
Pasture moves less water, so more of the sun's energy goes into heating the ground.
If so, pixels whose NDVI dropped should also show a rise in daytime land surface temperature.

**How**
- Daytime LST (Terra, about 10:30 am overpass) for the same dry-season months and the same before/after windows.
- Both windows end in 2022, before the orbit drift found in Phase 1.
- LST's native grid is about 5 km (0.05°). The 1 km NDVI change is averaged into 5 × 5 blocks so both share one grid.
  Each point in the scatter is then one ~5 km block.
- Requests: 2 regions × 2 windows × 3 years × 3 months = **36**.

**Why compare cleared and stable pixels:** the whole region may have warmed over 20 years for other reasons
(regional climate, drought years). The fair estimate of the clearing effect is

> ΔLST of cleared pixels − ΔLST of stable pixels

because whatever warmed both cancels out.

In [ ]:
LST_DEG = 0.05
K = round(LST_DEG / DEG)            # 5 NDVI pixels per LST pixel in each direction
LST_REGIONS = ["Rondônia", "Pará (BR-163)"]


def coarsen(a, k):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    return nanmean(a[:h, :w].reshape(h // k, k, w // k, k), axis=(1, 3))


lst_delta, ndvi_delta_c, lst_tmpl = {}, {}, {}
for n in LST_REGIONS:
    r = REGIONS[n]
    before = window_grid(LST, BASE_YEARS, r["months"], r["bbox"], LST_DEG)
    after = window_grid(LST, RECENT_YEARS, r["months"], r["bbox"], LST_DEG)
    lst_delta[n] = after.values - before.values            # K, which equals °C for a difference
    ndvi_delta_c[n] = coarsen(delta[n], K)
    lst_tmpl[n] = before
    assert lst_delta[n].shape == ndvi_delta_c[n].shape, (lst_delta[n].shape, ndvi_delta_c[n].shape)

n = "Rondônia"
t = np.nanpercentile(np.abs(lst_delta[n]), 98)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
im0 = show_map(axes[0], ndvi_delta_c[n], lst_tmpl[n].extent, title="NDVI change (5 km blocks)",
               cmap=DIFF_CMAP, vmin=-0.4, vmax=0.4)
im1 = show_map(axes[1], lst_delta[n], lst_tmpl[n].extent, title="Daytime land surface temperature change",
               cmap=TEMP_CMAP, vmin=-t, vmax=t)
fig.colorbar(im0, ax=axes[0], location="bottom", shrink=0.8, label="NDVI change")
fig.colorbar(im1, ax=axes[1], location="bottom", shrink=0.8, label="°C change")
fig.suptitle(f"{n}: dry season {BASE_YEARS[0]}–{BASE_YEARS[-1]} → {RECENT_YEARS[0]}–{RECENT_YEARS[-1]}",
             fontsize=12, x=0.02, ha="left", color=INK)
plt.show()

**Scatter of the two changes.** Each hexagon counts ~5 km blocks from both regions; darker = more blocks.
The black line is the median temperature change within each 0.05-wide NDVI-change bin.
- **If clearing warms the surface:** the line rises to the left, so bigger NDVI losses go with bigger warming.
- **If it doesn't:** the line stays flat.

In [ ]:
x = np.concatenate([ndvi_delta_c[n].ravel() for n in LST_REGIONS])
y = np.concatenate([lst_delta[n].ravel() for n in LST_REGIONS])
ok = np.isfinite(x) & np.isfinite(y)
x, y = x[ok], y[ok]

edges = np.arange(-0.6, 0.25, 0.05)
mid = (edges[:-1] + edges[1:]) / 2
idx = np.digitize(x, edges) - 1
med = np.array([np.median(y[idx == i]) if (idx == i).sum() >= 20 else np.nan for i in range(len(mid))])

fig, ax = plt.subplots(figsize=(8, 5.5), layout="constrained")
hb = ax.hexbin(x, y, gridsize=45, cmap="Blues", bins="log", mincnt=1, extent=(-0.6, 0.2, *np.percentile(y, [0.5, 99.5])))
ax.plot(mid, med, color=INK, lw=2, marker="o", ms=4, label="median per bin")
ax.axhline(0, color=AXIS, lw=1)
ax.axvline(-DROP, color=MUTED, lw=1, ls="--")
ax.text(-DROP - 0.01, 0.98, "cleared ←", transform=ax.get_xaxis_transform(), ha="right", va="top",
        fontsize=8, color=INK2)
ax.set_xlabel("Dry-season NDVI change (5 km block)")
ax.set_ylabel("Daytime land surface temperature change (°C)")
ax.legend(loc="upper right", fontsize=8)
fig.colorbar(hb, ax=ax, label="number of 5 km blocks (log scale)")
ax.set_title(f"Greenness loss vs. warming, {' + '.join(LST_REGIONS)}", fontsize=11)

plt.show()

effect = pd.DataFrame({
    n: {
        "ΔLST cleared (°C)": nanmean(lst_delta[n][ndvi_delta_c[n] < -DROP]),
        "ΔLST stable (°C)": nanmean(lst_delta[n][np.abs(ndvi_delta_c[n]) < 0.05]),
        "n cleared blocks": int((ndvi_delta_c[n] < -DROP).sum()),
    } for n in LST_REGIONS
}).T
effect["clearing effect (°C)"] = effect["ΔLST cleared (°C)"] - effect["ΔLST stable (°C)"]
effect.round(2)

## 11. Caveats and next steps

**Caveats to state in the project**
- **Resolution:** about 1 km pixels miss small clearings and smallholder plots. Area estimates are lower bounds
  on what Landsat-based (30 m) products report.
- **NDVI ≠ forest:** a drop can also come from fire, drought, or a cloud and smoke left in a composite.
  The persistence rule (§6) and the control region (§5, §7) limit false alarms, but can't remove them.
- **Regrowth and plantations:** oil palm and pulpwood plantations green up again after a few years.
  Riau's NDVI can then recover even though the original forest is gone, so this method undercounts conversion there.
- **Baseline:** clearing before 2004 is part of the baseline and isn't counted.
- **Binned values:** GIBS NDVI comes back in color-map bins. That is fine for changes of 0.2,
  but don't over-read differences smaller than about 0.01.
- **Orbit drift:** the analysis stops at 2022. §8 shows whether NDVI could safely go further.

**Possible next steps**
- **Land cover:** run `find_layers("land cover")`. An annual MODIS land-cover layer could show forest → pasture transitions directly.
  Its classes are color categories, though, so `decode()` currently turns them into NaN.
  It would need a small class decoder that returns `cmap.classes` labels.
- **True color:** run `find_layers("CorrectedReflectance")` for a before/after photo-like view.
  This is the most intuitive picture for a general audience.
- **Fire:** check `find_layers("fire")` or `find_layers("thermal")` to link clearing years to burning.